### Uso das técnicas de forecasting apresentadas em aula com Airline Passengers Dataset

#### Como o dataset aprendetado em aula (Jena Climate) possui 14 variáveis, nós precisamos adaptar o código para lidar com um dataset com apenas uma variável (número de passageiros).

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url)

print(df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(df["Passengers"])
plt.title("AirPassengers")
plt.xlabel("Mês")
plt.ylabel("Passageiros")
plt.show()

In [ ]:
values = df["Passengers"].values.astype("float32")

raw_data = values.reshape(-1, 1).copy()

raw_data.shape

In [ ]:
num_train_samples = int(0.7 * len(raw_data))
num_val_samples = int(0.15 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples

print(num_train_samples)
print(num_val_samples)
print(num_test_samples)

### Adaptação do código de preparação dos dados para o formato de séries temporais, utilizando a função `timeseries_dataset_from_array` do Keras retirada de https://github.com/fboldt/aulasann/blob/main/aula08a_forecasting.ipynb

In [ ]:
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean

std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

normalized_values = values.copy()
normalized_values = (normalized_values - mean[0]) / std[0]

In [ ]:
sampling_rate = 1
sequence_length = 12
delay = 1
batch_size = 16

In [ ]:
from tensorflow import keras

train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=normalized_values[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    shuffle=True,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples
)

val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=normalized_values[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples
)

test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=normalized_values[delay:],
    sequence_length=sequence_length,
    sampling_rate=sampling_rate,
    shuffle=False,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples
)

### Implementação do metodo ingênuo de previsão retirada de https://github.com/fboldt/aulasann/blob/main/aula08a_forecasting.ipynb

### Baseline

In [ ]:
import numpy as np

field = 0

def evaluate_naive_method(dataset):
    total_abs_err = 0
    samples_seen = 0
    for samples, targets in dataset:
        preds = samples[:, -1, field]
        total_abs_err += np.sum(np.abs(preds - targets))
        samples_seen += samples.shape[0]
    return total_abs_err / samples_seen
print(f"Validation MAE: {evaluate_naive_method(val_dataset):.2f}")
print(f"Test MAE: {evaluate_naive_method(test_dataset):.2f}")

### Densely connected

In [ ]:
from tensorflow.keras import layers

inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Flatten()(inputs)
x = layers.Dense(16, activation='relu')(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint('air_dense.keras', save_best_only=True)
]
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks
)
model = keras.models.load_model('air_dense.keras')
print(f'Test MAE: {model.evaluate(test_dataset)[1]:.2f}')

### 1D convolutional model

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Conv1D(8, 3, activation='relu')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 3, activation='relu')(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint('air_conv.keras', save_best_only=True)
]
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks
)
model = keras.models.load_model('air_conv.keras')
print(f'Test MAE: {model.evaluate(test_dataset)[1]:.2f}')

### LSTM

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(16)(inputs)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs=inputs, outputs=outputs)

callbacks = [keras.callbacks.ModelCheckpoint("air_lstm.keras", save_best_only=True)]

model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)
model = keras.models.load_model("air_lstm.keras")
print(f'Test MAE: {model.evaluate(test_dataset)[1]:.2f}')

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(16)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs=inputs, outputs=outputs)

callbacks = [keras.callbacks.ModelCheckpoint("air_lstm.keras", save_best_only=True)]

model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)
model = keras.models.load_model("air_lstm.keras")
print(f'Test MAE: {model.evaluate(test_dataset)[1]:.2f}')

### Stacking recurrent layers

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.GRU(32, recurrent_dropout=0.5, return_sequences=True)(inputs)
x = layers.GRU(32, recurrent_dropout=0.5)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs=inputs, outputs=outputs)

callbacks = [keras.callbacks.ModelCheckpoint("air_stacked_gru.keras", save_best_only=True)]

model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)
model = keras.models.load_model("air_stacked_gru.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")